# 09 - CRISPR-Cas9 deletion example in *Aspergillus oryzae*


This notebook is a toned-down CRISPR design example inspired by a more elaborate StreptoCAD workflow. The goal here is not to reproduce a full plasmid construction workflow, but to show how `teemi` can be used to:

- load a genome record
- identify candidate Cas9 guides for selected genes
- choose a small subset of guides per target
- fetch 45 bp upstream and downstream homology arms for a clean deletion strategy

To keep the example compact and readable, this notebook stops at design tables rather than building a full project export folder. For guide design and off-target screening we concatenate all eight *A. oryzae* RIB40 chromosomes into one combined genome record, while keeping the outputs dataframe-heavy.

The genome files are not bundled in the repository. Place `chrom1.gb` to `chrom8.gb` in `teemi_cad_workflows/inspiration_notebooks/data/09_aspergillus_crispr_example/` before running the notebook.


In [17]:
%%capture
import sys
if 'google.colab' in sys.modules:
    %pip install pydna teemi biopython pandas

## Imports

In [18]:
from pathlib import Path

import pandas as pd
from pydna.dseqrecord import Dseqrecord

from teemi.design.fetch_sequences import read_genbank_files, concatenate_genbank_records
from teemi.design.crispr_cas import SgRNAargs, extract_sgRNAs
from teemi.design.gibson_cloning import extract_locus_tag_homology_arms

## Input data

In [19]:
# This example uses all eight Aspergillus oryzae RIB40 chromosomes.
genome_dir = Path('../data/09_aspergillus_crispr_example')
chromosome_paths = sorted(genome_dir.glob('chrom*.gb'), key=lambda path: int(path.stem.replace('chrom', '')))

if len(chromosome_paths) != 8:
    raise FileNotFoundError(
        'Expected chrom1.gb to chrom8.gb in ../data/09_aspergillus_crispr_example/. '
        'Add the local Aspergillus genome files before running this notebook.'
    )

chromosome_records = [read_genbank_files(str(path))[0] for path in chromosome_paths]

chromosome_summary_df = pd.DataFrame([
    {
        'chromosome': record.id,
        'length_bp': len(record.seq),
        'feature_count': len(record.features),
    }
    for record in chromosome_records
])

chromosome_summary_df

,chromosome,length_bp,feature_count
0,BA000049.1,6520266,10184
1,BA000050.1,6264705,10054
2,BA000051.1,5123684,8291
3,BA000052.1,4887096,7596
4,BA000053.1,4533889,7378
5,BA000054.1,4190347,6562
6,BA000055.1,2933481,4571
7,BA000056.1,3395259,5397


In [20]:
combined_genome_record = concatenate_genbank_records(
    chromosome_records,
    record_id='A_oryzae_RIB40_combined',
    record_name='A_oryzae_RIB40_combined',
)
combined_genome = Dseqrecord(combined_genome_record)

pd.DataFrame([
    {
        'combined_record_id': combined_genome_record.id,
        'combined_length_bp': len(combined_genome_record.seq),
        'combined_feature_count': len(combined_genome_record.features),
    }
])

,combined_record_id,combined_length_bp,combined_feature_count
0,A_oryzae_RIB40_combined,37848727,60033


For a lightweight example, we use three early locus tags from chromosome 1. You can replace these with any other locus tags from the same GenBank file.

In [21]:
genes_to_delete = [
    'AO090009000001',
    'AO090009000002',
    'AO090009000003',
]

genes_to_delete

['AO090009000001', 'AO090009000002', 'AO090009000003']

## Find candidate Cas9 guides

In [22]:
args = SgRNAargs(
    combined_genome,
    genes_to_delete,
    cas_type='cas9',
    step=['find', 'filter'],
    gc_upper=0.72,
    gc_lower=0.35,
    off_target_seed=13,
    off_target_upper=5,
)

sgrna_df = extract_sgRNAs(args)
sgrna_df.head()

sgRNA generated were outside the designated border in AO090009000001. To incorporate this extent borders. Skipping to next locus tag.
sgRNA generated were outside the designated border in AO090009000001. To incorporate this extent borders. Skipping to next locus tag.
sgRNA generated were outside the designated border in AO090009000001. To incorporate this extent borders. Skipping to next locus tag.
sgRNA generated were outside the designated border in AO090009000002. To incorporate this extent borders. Skipping to next locus tag.
Pam was found outside designated locus_tag: AO090009000002. To incorporate this extent borders. Skipping to next locus tag.
sgRNA generated were outside the designated border in AO090009000002. To incorporate this extent borders. Skipping to next locus tag.
sgRNA generated were outside the designated border in AO090009000002. To incorporate this extent borders. Skipping to next locus tag.
sgRNA generated were outside the designated border in AO090009000003. To

,strain_name,locus_tag,gene_loc,gene_strand,sgrna_strand,sgrna_loc,gc,pam,sgrna,sgrna_seed_sequence,off_target_count
1503,A_oryzae_RIB40_combined,AO090009000002,9683,-1,-1,3241,0.45,AGG,TTATGCGAATGCTATTCGGC,AATGCTATTCGGC,0
1502,A_oryzae_RIB40_combined,AO090009000002,9683,-1,-1,3246,0.55,AGG,CGAATGCTATTCGGCAGGTG,TATTCGGCAGGTG,0
1501,A_oryzae_RIB40_combined,AO090009000002,9683,-1,-1,3247,0.50,GGG,GAATGCTATTCGGCAGGTGA,ATTCGGCAGGTGA,0
1500,A_oryzae_RIB40_combined,AO090009000002,9683,-1,-1,3253,0.50,TGG,TATTCGGCAGGTGAGGGAAT,CAGGTGAGGGAAT,0
1499,A_oryzae_RIB40_combined,AO090009000002,9683,-1,-1,3274,0.55,AGG,GGCCCAGATATTACAGTGCG,ATATTACAGTGCG,0


Here we keep the top two filtered guides per locus tag, ordered by off-target count.

In [23]:
selected_guides = (
    sgrna_df
    .sort_values(['locus_tag', 'off_target_count', 'gc'])
    .groupby('locus_tag')
    .head(2)
    .reset_index(drop=True)
)

selected_guides[['locus_tag', 'gene_loc', 'sgrna_loc', 'sgrna_strand', 'gc', 'pam', 'sgrna', 'off_target_count']]

,locus_tag,gene_loc,sgrna_loc,sgrna_strand,gc,pam,sgrna,off_target_count
0,AO090009000001,882,3764,-1,0.35,CGG,TCCAGTATATATTGCTGGAT,0
1,AO090009000001,882,3031,-1,0.35,CGG,AGACTGCAAATACATCCTTA,0
2,AO090009000002,9683,3329,-1,0.35,GGG,ATTTTACAACGATTCACGCA,0
3,AO090009000002,9683,3596,-1,0.35,TGG,GTATTAGATGGGTATTTGGA,0
4,AO090009000003,15692,2588,1,0.35,AGG,ATAATATATTGCCGCTTTCG,0
5,AO090009000003,15692,2115,-1,0.35,TGG,AGTTGGTGGTCTATTATCTT,0


## Fetch 45 bp homology arms for repair

For this example we keep the repair design minimal: a user-defined number of base pairs immediately upstream of the CDS plus the same number immediately downstream of the CDS. These are concatenated into one repair oligo for ordering.

In [ ]:
repair_arm_length = 45

homology_arm_df = extract_locus_tag_homology_arms(
    combined_genome_record,
    genes_to_delete,
    arm_length=repair_arm_length,
)

homology_arm_df

,locus_tag,gene_start,gene_end,gene_strand,upstream_arm_name,downstream_arm_name,repair_oligo_name,upstream_arm_start,upstream_arm_end,downstream_arm_start,downstream_arm_end,upstream_arm_length,downstream_arm_length,repair_oligo_length,upstream_arm,downstream_arm,repair_oligo
0,AO090009000001,881,5162,1,AO090009000001_UP30,AO090009000001_DW30,AO090009000001_DEL_30bp_arms,851,881,5162,5192,30,30,60,CCTCCCCCCCCCCCCCCGGAGGGCTGCGCC,TTTAGCTTATACAGTGTATATGAATATATA,CCTCCCCCCCCCCCCCCGGAGGGCTGCGCCTTTAGCTTATACAGTG...
1,AO090009000002,9682,14221,-1,AO090009000002_UP30,AO090009000002_DW30,AO090009000002_DEL_30bp_arms,9652,9682,14221,14251,30,30,60,TCTATATCTTGGATATACAACCTCGAATCC,GGTCTTTAAGCTCCGTCCAATATATTGACA,TCTATATCTTGGATATACAACCTCGAATCCGGTCTTTAAGCTCCGT...
2,AO090009000003,15691,18301,-1,AO090009000003_UP30,AO090009000003_DW30,AO090009000003_DEL_30bp_arms,15661,15691,18301,18331,30,30,60,CAATACAGTCGATTAAAGAAGATAGAATAA,CGAGATAGGATTTTTTACTTGCGGATCCAA,CAATACAGTCGATTAAAGAAGATAGAATAACGAGATAGGATTTTTT...


In [25]:
repair_design_df = homology_arm_df.assign(
    deletion_name=lambda df: df['locus_tag'] + '_DEL',
    repair_design_name=lambda df: df['repair_oligo_name'],
    deleted_interval=lambda df: df['gene_start'].astype(str) + ':' + df['gene_end'].astype(str),
)[[
    'locus_tag',
    'deletion_name',
    'repair_design_name',
    'gene_start',
    'gene_end',
    'gene_strand',
    'deleted_interval',
    'upstream_arm_name',
    'upstream_arm_start',
    'upstream_arm_end',
    'upstream_arm_length',
    'downstream_arm_name',
    'downstream_arm_start',
    'downstream_arm_end',
    'downstream_arm_length',
    'repair_oligo_length',
    'repair_oligo',
]]

repair_design_df

,locus_tag,deletion_name,repair_design_name,gene_start,gene_end,gene_strand,deleted_interval,upstream_arm_name,upstream_arm_start,upstream_arm_end,upstream_arm_length,downstream_arm_name,downstream_arm_start,downstream_arm_end,downstream_arm_length,repair_oligo_length,repair_oligo
0,AO090009000001,AO090009000001_DEL,AO090009000001_DEL_30bp_arms,881,5162,1,881:5162,AO090009000001_UP30,851,881,30,AO090009000001_DW30,5162,5192,30,60,CCTCCCCCCCCCCCCCCGGAGGGCTGCGCCTTTAGCTTATACAGTG...
1,AO090009000002,AO090009000002_DEL,AO090009000002_DEL_30bp_arms,9682,14221,-1,9682:14221,AO090009000002_UP30,9652,9682,30,AO090009000002_DW30,14221,14251,30,60,TCTATATCTTGGATATACAACCTCGAATCCGGTCTTTAAGCTCCGT...
2,AO090009000003,AO090009000003_DEL,AO090009000003_DEL_30bp_arms,15691,18301,-1,15691:18301,AO090009000003_UP30,15661,15691,30,AO090009000003_DW30,18301,18331,30,60,CAATACAGTCGATTAAAGAAGATAGAATAACGAGATAGGATTTTTT...


In [26]:
primary_guide_df = (
    selected_guides
    .sort_values(['locus_tag', 'off_target_count', 'gc'])
    .groupby('locus_tag')
    .head(1)
    .loc[:, ['locus_tag', 'sgrna', 'pam', 'off_target_count']]
    .rename(columns={'sgrna': 'primary_sgrna'})
    .reset_index(drop=True)
)

crispr_deletion_design_df = repair_design_df.merge(primary_guide_df, on='locus_tag', how='left')
crispr_deletion_design_df

,locus_tag,deletion_name,repair_design_name,gene_start,gene_end,gene_strand,deleted_interval,upstream_arm_name,upstream_arm_start,upstream_arm_end,upstream_arm_length,downstream_arm_name,downstream_arm_start,downstream_arm_end,downstream_arm_length,repair_oligo_length,repair_oligo,primary_sgrna,pam,off_target_count
0,AO090009000001,AO090009000001_DEL,AO090009000001_DEL_30bp_arms,881,5162,1,881:5162,AO090009000001_UP30,851,881,30,AO090009000001_DW30,5162,5192,30,60,CCTCCCCCCCCCCCCCCGGAGGGCTGCGCCTTTAGCTTATACAGTG...,TCCAGTATATATTGCTGGAT,CGG,0
1,AO090009000002,AO090009000002_DEL,AO090009000002_DEL_30bp_arms,9682,14221,-1,9682:14221,AO090009000002_UP30,9652,9682,30,AO090009000002_DW30,14221,14251,30,60,TCTATATCTTGGATATACAACCTCGAATCCGGTCTTTAAGCTCCGT...,ATTTTACAACGATTCACGCA,GGG,0
2,AO090009000003,AO090009000003_DEL,AO090009000003_DEL_30bp_arms,15691,18301,-1,15691:18301,AO090009000003_UP30,15661,15691,30,AO090009000003_DW30,18301,18331,30,60,CAATACAGTCGATTAAAGAAGATAGAATAACGAGATAGGATTTTTT...,ATAATATATTGCCGCTTTCG,AGG,0


## Example ordering table

In [27]:
guide_order_df = selected_guides.assign(
    oligo_name=lambda df: df['locus_tag'] + '_sgRNA_' + (df.groupby('locus_tag').cumcount() + 1).astype(str),
    item_type='sgRNA spacer',
    sequence=lambda df: df['sgrna'],
)[['oligo_name', 'item_type', 'sequence', 'locus_tag', 'off_target_count']]

repair_oligo_order_df = homology_arm_df.assign(
    oligo_name=lambda df: df['repair_oligo_name'],
    item_type='repair oligo',
    sequence=lambda df: df['repair_oligo'],
)[['oligo_name', 'item_type', 'sequence', 'locus_tag', 'repair_oligo_length']]

order_table = pd.concat(
    [guide_order_df, repair_oligo_order_df],
    ignore_index=True,
)
order_table

,oligo_name,item_type,sequence,locus_tag,off_target_count,repair_oligo_length
0,AO090009000001_sgRNA_1,sgRNA spacer,TCCAGTATATATTGCTGGAT,AO090009000001,0.0,NaN
1,AO090009000001_sgRNA_2,sgRNA spacer,AGACTGCAAATACATCCTTA,AO090009000001,0.0,NaN
2,AO090009000002_sgRNA_1,sgRNA spacer,ATTTTACAACGATTCACGCA,AO090009000002,0.0,NaN
3,AO090009000002_sgRNA_2,sgRNA spacer,GTATTAGATGGGTATTTGGA,AO090009000002,0.0,NaN
4,AO090009000003_sgRNA_1,sgRNA spacer,ATAATATATTGCCGCTTTCG,AO090009000003,0.0,NaN
5,AO090009000003_sgRNA_2,sgRNA spacer,AGTTGGTGGTCTATTATCTT,AO090009000003,0.0,NaN
6,AO090009000001_DEL_30bp_arms,repair oligo,CCTCCCCCCCCCCCCCCGGAGGGCTGCGCCTTTAGCTTATACAGTG...,AO090009000001,NaN,60.0
7,AO090009000002_DEL_30bp_arms,repair oligo,TCTATATCTTGGATATACAACCTCGAATCCGGTCTTTAAGCTCCGT...,AO090009000002,NaN,60.0
8,AO090009000003_DEL_30bp_arms,repair oligo,CAATACAGTCGATTAAAGAAGATAGAATAACGAGATAGGATTTTTT...,AO090009000003,NaN,60.0


## Eurofins-ready oligo table

In [28]:
eurofins_order_df = order_table.assign(
    Name=lambda df: df['oligo_name'],
    Sequence=lambda df: df['sequence'],
    Scale='25nm',
    Purification='Desalted',
    Form='Dry',
    Comments=lambda df: df['item_type'] + ' | ' + df['locus_tag'],
)[['Name', 'Sequence', 'Scale', 'Purification', 'Form', 'Comments']]

eurofins_order_df

,Name,Sequence,Scale,Purification,Form,Comments
0,AO090009000001_sgRNA_1,TCCAGTATATATTGCTGGAT,25nm,Desalted,Dry,sgRNA spacer | AO090009000001
1,AO090009000001_sgRNA_2,AGACTGCAAATACATCCTTA,25nm,Desalted,Dry,sgRNA spacer | AO090009000001
2,AO090009000002_sgRNA_1,ATTTTACAACGATTCACGCA,25nm,Desalted,Dry,sgRNA spacer | AO090009000002
3,AO090009000002_sgRNA_2,GTATTAGATGGGTATTTGGA,25nm,Desalted,Dry,sgRNA spacer | AO090009000002
4,AO090009000003_sgRNA_1,ATAATATATTGCCGCTTTCG,25nm,Desalted,Dry,sgRNA spacer | AO090009000003
5,AO090009000003_sgRNA_2,AGTTGGTGGTCTATTATCTT,25nm,Desalted,Dry,sgRNA spacer | AO090009000003
6,AO090009000001_DEL_30bp_arms,CCTCCCCCCCCCCCCCCGGAGGGCTGCGCCTTTAGCTTATACAGTG...,25nm,Desalted,Dry,repair oligo | AO090009000001
7,AO090009000002_DEL_30bp_arms,TCTATATCTTGGATATACAACCTCGAATCCGGTCTTTAAGCTCCGT...,25nm,Desalted,Dry,repair oligo | AO090009000002
8,AO090009000003_DEL_30bp_arms,CAATACAGTCGATTAAAGAAGATAGAATAACGAGATAGGATTTTTT...,25nm,Desalted,Dry,repair oligo | AO090009000003


This notebook is meant as a compact design example. A fuller workflow could extend this by:

- exporting the Eurofins table directly to CSV for ordering
- adding a fuller donor-construction strategy around the 45 bp arms
- generating validation primers for edited strains
- creating build-ready plate plans or project folders
- scaling the workflow across additional *A. oryzae* chromosomes or other fungi
